# Train Forward Neural Networks Using Related Work Data

In [ ]:
import os
from typing import Any

import lightning as pl
from hydra.utils import instantiate
from lightning.pytorch.callbacks import ModelCheckpoint, TQDMProgressBar
from omegaconf import DictConfig, OmegaConf, open_dict

## Define Generic Training Function

In [ ]:
def train_loop(cfg: DictConfig, seed: int = 42) -> None:
    """Train loop for Pytorch Lightning"""
    print(f"Training with following config:\n{OmegaConf.to_yaml(cfg)}")
    pl.seed_everything(seed)

    # Load data module
    data_module = instantiate(cfg.data_module, cfg=cfg)
    cfg = data_module.cfg

    # Save every two validation steps
    callback = ModelCheckpoint(
        monitor="val_loss",
        filename=f"{cfg.name}" + "-{epoch}-{val_loss:.4f}",
        every_n_epochs=2 * cfg.trainer.check_val_every_n_epoch,
    )
    # Progress bar callback
    bar = TQDMProgressBar(refresh_rate=cfg.progress_bar_refresh_rate)
    # Instantiate profiler
    profiler = instantiate(cfg.profiler)
    # Instantiate trainer
    trainer = pl.Trainer(
        **cfg.trainer,
        profiler=profiler,
        callbacks=[callback, bar],
    )

    # Instantiate model
    with trainer.init_module():
        model = instantiate(cfg.model, cfg=cfg, data_module=data_module)
    # Log amount of trainable network parameters for easier evaluation
    with open_dict(cfg):
        try:
            n_trainable_params = sum(
                p.numel() for p in model.generator.parameters() if p.requires_grad
            )
        except AttributeError:
            n_trainable_params = sum(
                p.numel() for p in model.parameters() if p.requires_grad
            )
        print("Number of trainable parameters: ", n_trainable_params)
        cfg.n_trainable_params = n_trainable_params

    # save OmegaConf in PyTorch Lightning/ Tensorboard structure
    save_dir = trainer.logger.log_dir
    print(f"Save dir set to: {save_dir}")
    os.makedirs(save_dir, exist_ok=True)
    OmegaConf.save(config=cfg, f=save_dir + "/config_log.yaml")

    # Training and validation
    print(f"Training path set to: {cfg.ckpt_path}")
    trainer.fit(model, datamodule=data_module, ckpt_path=cfg.ckpt_path)
    trainer.test(ckpt_path="best", datamodule=data_module)

### Define Default Parameters

In [ ]:
# Define often used parameters
# data
batch_size = 256
n_wvl = 1
n_params = 5
# model
train_mode = "train"
# optimizer
optimizer_lr = 1e-4
max_epochs = int(batch_size * 5)
patience = max_epochs // 40

### Define Base Config

In [ ]:
# define a generic config
def get_base_config(
    name: str,
    batch_size: int = batch_size,
    n_params: int = n_params,
    n_wvl: int = n_wvl,
    optimizer_lr: float = optimizer_lr,
    max_epochs: int = max_epochs,
    patience: int = patience,
    train_mode: str = train_mode,
    train_data_ratio: float = 1.0,
) -> dict[str, Any]:
    return {
        "train_mode": train_mode,
        "n_comp": None,
        "n_workers": 0,
        "batch_size": batch_size,
        "n_params": n_params,
        "max_augm_epochs": 0,
        "augm_prob": 0,
        "n_wavelengths": n_wvl,
        "train_data_ratio": train_data_ratio,
        "name": name,
        "model": {
            "_target_": "mcmlnet.training.models.base_model.BaseModel",
            "_recursive_": False,
            "mode": train_mode,
            "batch_size": batch_size,
            "max_augm_epochs": 0,
            "n_wavelengths": n_wvl,
            "generator": {
                "_target_": "mcmlnet.training.models.network_constructor.ForwardSurrogateModel",  # noqa: E501
                "_recursive_": False,
                "layers": [512, 512, 512, 512, 512, 1],
                "n_params": n_params,
                "arch_type": "minimal",
                "sigmoid": False,
                "p_dropout": 0.0,
            },
            "discriminator": None,
            "optimizer_gen": {
                "_target_": "mcmlnet.training.optimization.sophia_optimizer.SophiaG",
                "_recursive_": False,
                "lr": optimizer_lr,
                "betas": [0.965, 0.99],
                "rho": 0.1,
                "weight_decay": 0.001,
            },
            "scheduler_gen": {
                "_target_": "torch.optim.lr_scheduler.ReduceLROnPlateau",
                "patience": patience,
                "factor": 0.5,
                "threshold": 0.01,
                "threshold_mode": "rel",
                "min_lr": 1e-8,
            },
            "optimizer_dis": None,
            "scheduler_dis": None,
            "network_init": {
                "activation": "leaky_relu",
                "negative_slope": 0.2,
                "weight_init": "kaiming_normal",
                "bias_init": "zeros",
            },
            "loss_init": {
                "validate": {
                    "_target_": "torch.nn.functional.l1_loss",
                    "_recursive_": False,
                    "reduction": "mean",
                },
                "train": {
                    "_target_": "torch.nn.functional.l1_loss",
                    "_recursive_": False,
                    "reduction": "mean",
                },
            },
        },
        "preprocessing": {
            "_target_": "mcmlnet.training.data_loading.sota_data_classes.SOTAPreprocessor",  # noqa: E501
            "_recursive_": False,
            "dataset_name": name,
            "n_wavelengths": n_wvl,
            "n_pca_comp": None,
            "log_intensity": False,
            "kfolds": None,
            "fold": 0,
        },
        "data_module": {
            "_target_": "mcmlnet.training.data_loading.data_module.DataModule",
            "_recursive_": False,
        },
        "dataset": {
            "_target_": "mcmlnet.training.data_loading.sota_data_classes.SOTADataset",
            "_recursive_": False,
            "n_wavelengths": n_wvl,
            "n_params": n_params,
            "thick_deepest_layer": True,
        },
        "input_augmentations": {
            "_recursive_": False,
        },
        "output_augmentations": {
            "_recursive_": False,
        },
        "progress_bar_refresh_rate": 50,
        "ckpt_path": None,
        "trainer": {
            "accelerator": "gpu",
            "devices": 1,
            "check_val_every_n_epoch": patience,
            "max_epochs": max_epochs,
            "precision": "32-true",
            "benchmark": True,
        },
        "profiler": None,
        "checkpointing": {
            "monitor": "val_loss",
            "save_last": True,
            "save_top_k": 1,
            "mode": "min",
        },
    }

## Lan et al. 2023

> The overall NMAE of all predictions is 0.018.

- Caption of Figure 3, https://doi.org/10.1364/BOE.490164.

NMAE values are ranging between 0.0093 and 0.0196 for a train size of 95%

- Additional figures and data, https://opticapublishing.figshare.com/articles/dataset/NNFM_with_varying_training_size/23500770?file=41213445

### LHS

In [ ]:
# run training
lan_lhs_cfg = get_base_config("lan_lhs", max_epochs=2560, patience=64)
train_loop(OmegaConf.create(lan_lhs_cfg))

### Uniform

In [ ]:
# run training
lan_cfg = get_base_config("lan", max_epochs=2560, patience=64)
train_loop(OmegaConf.create(lan_cfg))

## Tsui et al. 2018

> The errors of the best models were 1.28%±1.19%, 2.27%±2.40% and 3.59%±4.27% for the SDS of 0.22 mm, 0.45 mm and 0.73 mm, respectively.

- Chapter 3, Results, https://doi.org/10.1364/BOE.9.001531

In [ ]:
# run training
tsui_cfg = get_base_config("tsui", n_params=20, max_epochs=2560, patience=64)
train_loop(OmegaConf.create(tsui_cfg))

## Manojlovic et al. 2025

No results of the F-NN, closest is the:

> Comparison of the forward fitting model MAE on simulated spectra.
>
> ...
>
> RFF + BNN + F-NN	0.0077

- Table 4, https://doi.org/10.1117/1.JBO.30.1.016004

In [ ]:
# run training
manoj_cfg = get_base_config(
    "manoj",
    n_params=10,
    n_wvl=351,
    patience=max_epochs // 40,
)
train_loop(OmegaConf.create(manoj_cfg))